In [ ]:
------------------------------------------------------------------------------------------------
DataScienceCourse/
├── notebooks/
│   ├── 00_template.ipynb           # Template, for copying and having relative paths functional.
│   ├── 01_limpieza_grupo1.ipynb
│   └── 05_eda_analisis.ipynb
├── src/
│   ├── __init__.py
│   ├── paths.py                    # Pathing Setup script
│   └── descriptive_tools.py        # Script datascience ds3on1, vartype, statsgroup
└── main.py                         # Entry point for the scripts./
        "ROOT": ROOT_DIR,
        "DATA": DATA_DIR,
        "TABLES": TABLES_DIR,
        "PROCESSED": PROCESSED_DATA_DIR
        "MODELS": MODELS_DIR
------------------------------------------------------------------------------------------------

In [17]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1, TABLES_DIR

# 1. Inicializar workspace y rutas
paths = setup_workspace()

# 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
# df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
# ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Workspace anclado en: /home/josu/Documentos/DataScienceCourse
📂 Ruta de tablas verificada: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


### 🟢 `< Module 10.2 >` Interactive Socio-Political & Macroeconomic Time Series Dashboard (Plotly Dash)

#### 🎯 Context & Objectives

While Module 10.1 demonstrated decoupled client-server REST inference on cross-sectional tabular data, real-world Data Science teams frequently build self-contained, reactive analytical backends to model temporal dynamics and run exogenous scenario simulations without writing separate HTML/JavaScript frontends.

In this module, you will build a reactive multi-tab web application using **Plotly Dash** (`src/dash_timeseries.py`) focused on Spain's longitudinal socio-political trajectory (`2002–2025`). Because biennial European Social Survey (ESS) rounds (`essround` 1–11) alone provide ~11 sparse observations, you will construct a higher-frequency quarterly time series by anchoring empirical ESS `lrscale` waves (`cntry == 'ES'`) with **population-weighted Spanish National and Regional (_Autonómicas_) electoral ideological gravity centers**, paired with a synchronized **synthetic/calibrated macroeconomic feature array** (`gdp_per_capita`, `inflation`, `unemployment`).

#### 📑 Exercise Instructions

##### 1. Temporal Dataset Engineering & Leakage-Free Preprocessing (`notebooks/Ex10.2_ts_exploration.ipynb`)

- **Party-Level Ideological Calibration (ESS Microdata):**
    
    - Query your ESS microdata for Spain (`cntry == 'ES'`) and compute each national and regional party's empirical mean `lrscale` ($LR_p \in [0, 10]$) after filtering non-response codes (`77`, `88`, `99`).
        
- **National & Regional (_Autonómicas_) Electoral Gravity Index (`lrscale_float`):**
    
    - Aggregate historical Spanish General Elections (national vote shares $s_{p,t}$) and Regional Elections (_Autonómicas_ across NUTS-2 `REGIONES` communities) into a quarterly series (`freq='QS'`).
        
    - For regional elections in community $r$ with census weight $w_r = \text{Pop}_r / \text{Pop}_{\text{ES}}$, update the national ideological index proportionally without introducing single-region spikes:
        

$$LR_{\text{ES}, t} = (1 - w_r) \cdot LR_{\text{ES}, t-1} + w_r \cdot LR_{r, t}$$

- Merge the empirical ESS `lrscale` wave averages as ground-truth calibration anchors.
    
- **Synthetic Macroeconomic Exogenous Array ($X_{\text{macro}}$):**
    
    - Generate a synchronized quarterly NumPy/Pandas array aligned with the electoral timeline containing three macro indicators calibrated to Spanish economic cycles (`2002–2025`): **GDP per capita (`gdp_per_capita`)**, **Inflation rate (`inflation_pct`)**, and **Unemployment rate (`unemployment_pct`)**.
        
- **Chronological Split & Inside-the-Fold Interpolation:**
    
    - Partition the series chronologically (e.g., `train` up to `2021-Q4`, `validation` from `2022-Q1` onward).
        
    - **Strict Anti-Leakage Rule:** Perform linear/spline interpolation of `lrscale_float` **exclusively inside the training split** using only past/current training anchors (never looking ahead into validation anchors), and evaluate forecasting errors strictly against real non-interpolated anchors in the validation window.
        

##### 2. Reactive Plotly Dash Backend (`src/dash_timeseries.py`)

Structure `src/dash_timeseries.py` into **three reactive tabs** driven by Dash callbacks (`@callback`):

1. **Tab 1 — Socio-Political Temporal EDA & Stationarity Diagnostics:**
    
    - Render an interactive Plotly dual-axis or synchronized chart displaying `lrscale_float` (distinguishing real ESS/electoral anchors from training-interpolated points) alongside the synthetic macroeconomic array (`gdp_per_capita`, `inflation_pct`, `unemployment_pct`).
        
    - Include a `dcc.RangeSlider` and a rolling-window slider ($k \in [2, 12]$ quarters) to overlay **Rolling Mean** and **Rolling Standard Deviation** dynamically.
        
    - Implement a `dcc.RadioItems` toggle switching between the **Original Series** ($y_t$) and **First-Order Differencing** ($\Delta y_t = y_t - y_{t-1}$), displaying in real time the **Augmented Dickey-Fuller (ADF) Test** (`statsmodels.tsa.stattools.adfuller`) statistic, $p$-value, critical values, and automated stationarity verdict ($p < 0.05$).
        
2. **Tab 2 — Seasonal & Trend Decomposition:**
    
    - Apply `statsmodels.tsa.seasonal.seasonal_decompose` (`period=4` for quarterly cycles) and render a synchronized 4-panel subplot (**Observed**, **Trend**, **Seasonal**, **Residual**).
        
    - Connect a `dcc.RadioItems` selector allowing the user to switch between **Additive** ($Y_t = T_t + S_t + R_t$) and **Multiplicative** ($Y_t = T_t \times S_t \times R_t$) decomposition models and inspect residual autocorrelation.
        
3. **Tab 3 — Exogenous `SARIMAX` Forecasting & Macro Stress-Test Simulator:**
    
    - Fit a seasonal exogenous model (`SARIMAX(endog=lrscale_float, exog=X_macro)`) strictly on the historical training split.
        
    - Provide a forecast horizon slider ($h \in [4, 16]$ quarters) plus **interactive macro scenario sliders** ($\Delta \text{GDP per capita}$, $\Delta \text{Inflation}$, $\Delta \text{Unemployment}$) that project future `lrscale_float` trajectories along with **80% and 95% confidence intervals** (`get_forecast().conf_int()`), overlaying the held-out validation ground truth.
        
    - Render a live metrics card computing **RMSE**, **MAE**, **MAPE**, and **AIC** over the overlapping validation anchors.
        

#### 📚 Official Documentation & Reference Sheets

- **Plotly Dash Core Components & Callbacks:** [https://dash.plotly.com/basic-callbacks](https://dash.plotly.com/basic-callbacks)
    
- **Statsmodels Time Series Analysis (`tsa`):** [https://www.statsmodels.org/stable/tsa.html](https://www.statsmodels.org/stable/tsa.html)
    
- **Augmented Dickey-Fuller (`adfuller`):** [https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html)
    
- **SARIMAX Exogenous Forecasting & Confidence Intervals:** [https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAXResults.get_forecast.html](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAXResults.get_forecast.html)
    

#### ✅ Deliverables Checklist

1. `notebooks/Ex10.2_ts_exploration.ipynb` (calibration of party `lrscale` from ESS, electoral aggregation, synthetic macro array generation, ADF tests, and `SARIMAX` validation).
    
2. `src/dash_timeseries.py` (standalone Dash backend running cleanly on `[http://127.0.0.1:8050](http://127.0.0.1:8050)`).
    
3. Rendered Markdown summary in `docs/Corrections/Ex10.2.md` linked from `docs/index.md`.
    

Cuando tengas listo el bloque de construcción del dataset en `notebooks/Ex10.2_ts_exploration.ipynb` (la tabla de calibración de partidos + anclas electorales + el array sintético de `gdp_per_capita`, `inflation_pct`, `unemployment_pct`), compártelo por aquí y lo auditamos antes de encapsularlo en `src/dash_timeseries.py`.

In [18]:
import pandas as pd
import numpy as np
import random
import duckdb as db

dftest = pd.read_csv(TABLES_DIR/'SpanishLrscale.csv')

In [21]:
dftest['name'].nunique()

11

In [22]:
# We come from "https://ess.sikt.no/en/data-builder/", before you select the rounds you can see in temporary tip the year of the round:
# Proddate is the year they publish the data, not the day they recorded it.
# ESS1 ROWS ARE FROM 2002. ESS2=2004, ESS3 2006, ESS4  2008,ESS5 2010,  ESS6 2012, ESS7 2014, ESS8 2016, ESS9 2018 ESS10SC 2020 ESS11 2023
#Lets make a new dataframe using year as one column and lrscale mean of every row sharing the same year=Ess round name:
ESS_YEARS = {
    1: 2002, 2: 2004, 3: 2006, 4: 2008, 5: 2010,
    6: 2012, 7: 2014, 8: 2016, 9: 2018, 10: 2020, 11: 2023
}

df_ess_es = (
    dftest.assign(
        round_num=dftest["name"].str.extract(r"^ESS(\d+)").astype(int),
        lrscale_clean=dftest["lrscale"].where(dftest["lrscale"].between(0, 10))
    )
    .assign(year=lambda d: d["round_num"].map(ESS_YEARS))
    .groupby("year", as_index=False)["lrscale_clean"]
    .mean()
    .rename(columns={"lrscale_clean": "lrscale"})
    .sort_values("year")
    .reset_index(drop=True)
)

print(df_ess_es)

    year   lrscale
0   2002  4.423049
1   2004  4.395464
2   2006  4.503090
3   2008  4.544487
4   2010  4.640212
5   2012  4.539792
6   2014  4.415162
7   2016  4.472918
8   2018  4.431834
9   2020  4.018544
10  2023  4.654599


In [26]:
# We will use the national electoral results and those of autonomous communities (only the most representative Catalonia, Madrid, the Basque Country 
# and Andalusia, big 4)
# as results for the scale of the missing years,
# using an estimated catalog of scale for each party and making a weighted average.

import pandas as pd
import numpy as np

# 1. Catálogo ideológico calibrado por partido (0 = Extrema Izquierda, 10 = Extrema Derecha)
PARTY_LR = {
    "IU_PODEMOS_SUMAR_ADELANTE": 2.3,
    "ERC_BILDU_CUP": 2.4,
    "PSOE_PSC_PSE": 3.8,
    "COMPROMIS_MASMADRID": 3.1,
    "PNV": 5.3,
    "CIU_JUNTS": 5.8,
    "CS_UPYD": 5.7,
    "PP": 7.2,
    "VOX": 8.6,
}

def calc_lr(shares: dict) -> float:
    valid = {k: v for k, v in shares.items() if k in PARTY_LR and v > 0}
    total = sum(valid.values())
    return sum((v / total) * PARTY_LR[k] for k, v in valid.items())

# 2. Pesos demográficos normalizados del Big 4 (Andalucía, Cataluña, Madrid, País Vasco)
BIG4_WEIGHTS = {
    "ANDALUCIA": 0.337,  # ~8.5M hab
    "CATALUNA":  0.307,  # ~7.9M hab
    "MADRID":    0.269,  # ~6.9M hab
    "PAIS_VASCO":0.087,  # ~2.2M hab
}

# 3. Historial electoral real del Big 4 (2003-2024) -> cada evento actualiza su comunidad
BIG4_ELECTIONS = [
    # 2003
    ("2003-04-01", "MADRID",    {"PP": 48.5, "PSOE_PSC_PSE": 39.0, "IU_PODEMOS_SUMAR_ADELANTE": 8.5}),
    ("2003-10-01", "CATALUNA",  {"CIU_JUNTS": 30.9, "PSOE_PSC_PSE": 31.2, "ERC_BILDU_CUP": 16.4, "PP": 11.9, "IU_PODEMOS_SUMAR_ADELANTE": 7.3}),
    # 2004
    ("2004-01-01", "ANDALUCIA", {"PSOE_PSC_PSE": 50.4, "PP": 31.8, "IU_PODEMOS_SUMAR_ADELANTE": 7.5}),
    # 2005
    ("2005-04-01", "PAIS_VASCO",{"PNV": 38.7, "PSOE_PSC_PSE": 22.7, "PP": 17.4, "ERC_BILDU_CUP": 12.4}),
    # 2006
    ("2006-10-01", "CATALUNA",  {"CIU_JUNTS": 31.5, "PSOE_PSC_PSE": 26.8, "ERC_BILDU_CUP": 14.0, "PP": 10.7, "IU_PODEMOS_SUMAR_ADELANTE": 9.5, "CS_UPYD": 3.0}),
    # 2007
    ("2007-04-01", "MADRID",    {"PP": 53.3, "PSOE_PSC_PSE": 33.6, "IU_PODEMOS_SUMAR_ADELANTE": 8.9}),
    # 2008
    ("2008-01-01", "ANDALUCIA", {"PSOE_PSC_PSE": 48.4, "PP": 38.5, "IU_PODEMOS_SUMAR_ADELANTE": 7.1}),
    # 2009
    ("2009-01-01", "PAIS_VASCO",{"PNV": 38.6, "PSOE_PSC_PSE": 30.7, "PP": 14.1, "ERC_BILDU_CUP": 6.0}),
    # 2010
    ("2010-10-01", "CATALUNA",  {"CIU_JUNTS": 38.4, "PSOE_PSC_PSE": 18.4, "PP": 12.4, "IU_PODEMOS_SUMAR_ADELANTE": 7.4, "ERC_BILDU_CUP": 7.0, "CS_UPYD": 3.4}),
    # 2011
    ("2011-04-01", "MADRID",    {"PP": 51.7, "PSOE_PSC_PSE": 26.3, "IU_PODEMOS_SUMAR_ADELANTE": 9.6, "CS_UPYD": 6.3}),
    # 2012
    ("2012-01-01", "ANDALUCIA", {"PP": 40.7, "PSOE_PSC_PSE": 39.6, "IU_PODEMOS_SUMAR_ADELANTE": 11.3}),
    ("2012-10-01", "PAIS_VASCO",{"PNV": 34.6, "ERC_BILDU_CUP": 25.0, "PSOE_PSC_PSE": 19.1, "PP": 11.7}),
    ("2012-10-01", "CATALUNA",  {"CIU_JUNTS": 30.7, "ERC_BILDU_CUP": 13.7, "PSOE_PSC_PSE": 14.4, "PP": 13.0, "IU_PODEMOS_SUMAR_ADELANTE": 9.9, "CS_UPYD": 7.6}),
    # 2015
    ("2015-01-01", "ANDALUCIA", {"PSOE_PSC_PSE": 35.4, "PP": 26.7, "IU_PODEMOS_SUMAR_ADELANTE": 21.7, "CS_UPYD": 9.3}),
    ("2015-04-01", "MADRID",    {"PP": 33.1, "PSOE_PSC_PSE": 25.4, "IU_PODEMOS_SUMAR_ADELANTE": 18.6, "CS_UPYD": 12.2}),
    ("2015-07-01", "CATALUNA",  {"CIU_JUNTS": 39.6, "CS_UPYD": 17.9, "PSOE_PSC_PSE": 12.7, "IU_PODEMOS_SUMAR_ADELANTE": 8.9, "PP": 8.5, "ERC_BILDU_CUP": 8.2}),
    # 2016
    ("2016-07-01", "PAIS_VASCO",{"PNV": 37.6, "ERC_BILDU_CUP": 21.3, "IU_PODEMOS_SUMAR_ADELANTE": 14.9, "PSOE_PSC_PSE": 11.9, "PP": 10.2}),
    # 2017
    ("2017-10-01", "CATALUNA",  {"CS_UPYD": 25.4, "CIU_JUNTS": 21.7, "ERC_BILDU_CUP": 21.4, "PSOE_PSC_PSE": 13.9, "IU_PODEMOS_SUMAR_ADELANTE": 7.5, "PP": 4.2}),
    # 2018
    ("2018-10-01", "ANDALUCIA", {"PSOE_PSC_PSE": 27.9, "PP": 20.7, "CS_UPYD": 18.3, "IU_PODEMOS_SUMAR_ADELANTE": 16.2, "VOX": 11.0}),
    # 2019
    ("2019-04-01", "MADRID",    {"PSOE_PSC_PSE": 27.3, "PP": 22.2, "CS_UPYD": 19.5, "COMPROMIS_MASMADRID": 14.7, "VOX": 8.9, "IU_PODEMOS_SUMAR_ADELANTE": 5.6}),
    # 2020
    ("2020-07-01", "PAIS_VASCO",{"PNV": 39.1, "ERC_BILDU_CUP": 27.9, "PSOE_PSC_PSE": 13.6, "IU_PODEMOS_SUMAR_ADELANTE": 8.1, "PP": 6.8}),
    # 2021
    ("2021-01-01", "CATALUNA",  {"PSOE_PSC_PSE": 23.0, "ERC_BILDU_CUP": 21.3, "CIU_JUNTS": 20.1, "VOX": 7.7, "IU_PODEMOS_SUMAR_ADELANTE": 6.9, "ERC_BILDU_CUP": 6.7, "CS_UPYD": 5.6, "PP": 3.8}),
    ("2021-04-01", "MADRID",    {"PP": 44.8, "COMPROMIS_MASMADRID": 17.0, "PSOE_PSC_PSE": 16.8, "VOX": 9.1, "IU_PODEMOS_SUMAR_ADELANTE": 7.2}),
    # 2022
    ("2022-04-01", "ANDALUCIA", {"PP": 43.1, "PSOE_PSC_PSE": 24.1, "VOX": 13.5, "IU_PODEMOS_SUMAR_ADELANTE": 12.3}),
    # 2023
    ("2023-04-01", "MADRID",    {"PP": 47.3, "COMPROMIS_MASMADRID": 18.4, "PSOE_PSC_PSE": 18.2, "VOX": 7.3, "IU_PODEMOS_SUMAR_ADELANTE": 4.8}),
    # 2024
    ("2024-04-01", "PAIS_VASCO",{"PNV": 35.2, "ERC_BILDU_CUP": 32.5, "PSOE_PSC_PSE": 14.2, "PP": 9.2, "IU_PODEMOS_SUMAR_ADELANTE": 5.6}),
    ("2024-04-01", "CATALUNA",  {"PSOE_PSC_PSE": 28.0, "CIU_JUNTS": 21.6, "ERC_BILDU_CUP": 13.7, "PP": 11.0, "VOX": 8.0, "IU_PODEMOS_SUMAR_ADELANTE": 5.8}),
]

# 4. Elecciones Generales de España (2004-2023) -> peso nacional directo
GENERAL_ELECTIONS = [
    ("2004-01-01", {"PSOE_PSC_PSE": 42.6, "PP": 37.7, "IU_PODEMOS_SUMAR_ADELANTE": 5.0, "CIU_JUNTS": 3.2, "ERC_BILDU_CUP": 2.5, "PNV": 1.6}),
    ("2008-01-01", {"PSOE_PSC_PSE": 43.9, "PP": 39.9, "IU_PODEMOS_SUMAR_ADELANTE": 3.8, "CIU_JUNTS": 3.0, "PNV": 1.2, "ERC_BILDU_CUP": 1.2, "CS_UPYD": 1.2}),
    ("2011-10-01", {"PP": 44.6, "PSOE_PSC_PSE": 28.8, "IU_PODEMOS_SUMAR_ADELANTE": 6.9, "CS_UPYD": 4.7, "CIU_JUNTS": 4.2, "ERC_BILDU_CUP": 2.4, "PNV": 1.3}),
    ("2015-10-01", {"PP": 28.7, "PSOE_PSC_PSE": 22.0, "IU_PODEMOS_SUMAR_ADELANTE": 24.4, "CS_UPYD": 13.9, "ERC_BILDU_CUP": 3.3, "CIU_JUNTS": 2.2, "PNV": 1.2}),
    ("2016-04-01", {"PP": 33.0, "PSOE_PSC_PSE": 22.6, "IU_PODEMOS_SUMAR_ADELANTE": 21.2, "CS_UPYD": 13.1, "ERC_BILDU_CUP": 3.4, "CIU_JUNTS": 2.0, "PNV": 1.2}),
    ("2019-04-01", {"PSOE_PSC_PSE": 28.7, "PP": 16.7, "CS_UPYD": 15.9, "IU_PODEMOS_SUMAR_ADELANTE": 14.3, "VOX": 10.3, "ERC_BILDU_CUP": 4.9, "CIU_JUNTS": 1.9, "PNV": 1.5}),
    ("2019-10-01", {"PSOE_PSC_PSE": 28.0, "PP": 20.8, "VOX": 15.1, "IU_PODEMOS_SUMAR_ADELANTE": 12.9, "CS_UPYD": 6.8, "ERC_BILDU_CUP": 4.7, "CIU_JUNTS": 2.2, "PNV": 1.6}),
    ("2023-07-01", {"PP": 33.1, "PSOE_PSC_PSE": 31.7, "VOX": 12.4, "IU_PODEMOS_SUMAR_ADELANTE": 12.3, "ERC_BILDU_CUP": 3.3, "CIU_JUNTS": 1.6, "PNV": 1.1}),
]

# 5. Calcular serie ponderada del Big 4 en cada fecha de elección autonómica
big4_state = {"ANDALUCIA": 4.65, "CATALUNA": 4.35, "MADRID": 5.30, "PAIS_VASCO": 4.30}  # Estado inicial 2002
big4_records = {}

for date_str, region, shares in BIG4_ELECTIONS:
    big4_state[region] = calc_lr(shares)
    weighted_mean = sum(BIG4_WEIGHTS[r] * big4_state[r] for r in BIG4_WEIGHTS)
    big4_records[pd.to_datetime(date_str)] = weighted_mean

# 6. Ensamblar índice trimestral 2002-Q1 a 2025-Q4 combinando ESS + Generales + Big 4
idx = pd.date_range("2002-01-01", "2025-10-01", freq="QS")
ts_df = pd.DataFrame(index=idx)
ts_df.index.name = "date"

ess_series = pd.Series(
    [4.423049, 4.395464, 4.503090, 4.544487, 4.640212, 4.539792, 4.415162, 4.472918, 4.431834, 4.018544, 4.654599],
    index=pd.to_datetime(["2002-07-01", "2004-07-01", "2006-07-01", "2008-07-01", "2010-07-01",
                          "2012-07-01", "2014-07-01", "2016-07-01", "2018-07-01", "2020-07-01", "2023-07-01"]),
    name="lr_ess"
)
gen_series = pd.Series({pd.to_datetime(d): calc_lr(s) for d, s in GENERAL_ELECTIONS}, name="lr_general")
big4_series = pd.Series(big4_records, name="lr_big4")

ts_df = ts_df.join([ess_series, gen_series, big4_series])

# Prioridad de anclaje: ESS (encuesta directa) > Generales > Media ponderada Big 4 Autonómicas
ts_df["lr_anchor"] = ts_df["lr_ess"].combine_first(ts_df["lr_general"]).combine_first(ts_df["lr_big4"])
ts_df["anchor_source"] = np.where(
    ts_df["lr_ess"].notna(), "ESS_Wave",
    np.where(ts_df["lr_general"].notna(), "General_Election",
    np.where(ts_df["lr_big4"].notna(), "Big4_Regional_Weighted", "Interpolated"))
)

print("Anclas reales obtenidas por fuente:")
print(ts_df["anchor_source"].value_counts())
print("\nMuestra de anclas Big 4 y Generales:")
print(ts_df[ts_df["anchor_source"] != "Interpolated"][["lr_anchor", "anchor_source"]].head(15))

Anclas reales obtenidas por fuente:
anchor_source
Interpolated              58
Big4_Regional_Weighted    20
ESS_Wave                  11
General_Election           7
Name: count, dtype: int64

Muestra de anclas Big 4 y Generales:
            lr_anchor           anchor_source
date                                         
2002-07-01   4.423049                ESS_Wave
2003-04-01   4.725137  Big4_Regional_Weighted
2003-10-01   4.771061  Big4_Regional_Weighted
2004-01-01   5.160475        General_Election
2004-07-01   4.395464                ESS_Wave
2005-04-01   4.900300  Big4_Regional_Weighted
2006-07-01   4.503090                ESS_Wave
2006-10-01   4.914506  Big4_Regional_Weighted
2007-04-01   4.959537  Big4_Regional_Weighted
2008-01-01   5.268790        General_Election
2008-07-01   4.544487                ESS_Wave
2009-01-01   5.026281  Big4_Regional_Weighted
2010-07-01   4.640212                ESS_Wave
2010-10-01   5.166124  Big4_Regional_Weighted
2011-04-01   5.191361  Big4_Region

In [27]:
ts_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 96 entries, 2002-01-01 to 2025-10-01
Freq: QS-JAN
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   lr_ess         11 non-null     float64
 1   lr_general     8 non-null      float64
 2   lr_big4        25 non-null     float64
 3   lr_anchor      38 non-null     float64
 4   anchor_source  96 non-null     str    
dtypes: float64(4), str(1)
memory usage: 7.9 KB


In [30]:
ts_df['lr_anchor'].head(36)

date
2002-01-01         NaN
2002-04-01         NaN
2002-07-01    4.423049
2002-10-01         NaN
2003-01-01         NaN
2003-04-01    4.725137
2003-07-01         NaN
2003-10-01    4.771061
2004-01-01    5.160475
2004-04-01         NaN
2004-07-01    4.395464
2004-10-01         NaN
2005-01-01         NaN
2005-04-01    4.900300
2005-07-01         NaN
2005-10-01         NaN
2006-01-01         NaN
2006-04-01         NaN
2006-07-01    4.503090
2006-10-01    4.914506
2007-01-01         NaN
2007-04-01    4.959537
2007-07-01         NaN
2007-10-01         NaN
2008-01-01    5.268790
2008-04-01         NaN
2008-07-01    4.544487
2008-10-01         NaN
2009-01-01    5.026281
2009-04-01         NaN
2009-07-01         NaN
2009-10-01         NaN
2010-01-01         NaN
2010-04-01         NaN
2010-07-01    4.640212
2010-10-01    5.166124
Freq: QS-JAN, Name: lr_anchor, dtype: float64

In [31]:
from statsmodels.tsa.stattools import adfuller

# 1. Visual continuous series (all 96 quarters) using shape-preserving PCHIP interpolation
ts_df["lrscale_pchip"] = ts_df["lr_anchor"].interpolate(method="pchip").bfill().ffill()

# 2. Strict chronological split (Train: 2002-Q1..2021-Q4 | Validation: 2022-Q1..2025-Q4)
train_df = ts_df.loc[:"2021-10-01"].copy()
val_df = ts_df.loc["2022-01-01":].copy()

# Interpolate strictly INSIDE train_df so validation anchors never leak backward
train_df["lrscale_float"] = train_df["lr_anchor"].interpolate(method="pchip").bfill().ffill()

# 3. Inspect gap distribution and stationarity (ADF test)
print("Summary of interpolated train series:")
print(train_df["lrscale_float"].describe().round(4))

for name, s in [
    ("Original Train Series (y_t)", train_df["lrscale_float"]),
    ("First Difference (Δy_t)", train_df["lrscale_float"].diff().dropna()),
]:
    stat, pval, _, _, crit, _ = adfuller(s)
    status = "STATIONARY (p < 0.05)" if pval < 0.05 else "NON-STATIONARY (p >= 0.05)"
    print(f"[{name}] ADF={stat:.4f} | p={pval:.4f} | 5% crit={crit['5%']:.4f} -> {status}")

Summary of interpolated train series:
count    80.0000
mean      4.7806
std       0.2927
min       3.9267
25%       4.5471
50%       4.7753
75%       4.9595
max       5.4922
Name: lrscale_float, dtype: float64
[Original Train Series (y_t)] ADF=-1.3070 | p=0.6260 | 5% crit=-2.9058 -> NON-STATIONARY (p >= 0.05)
[First Difference (Δy_t)] ADF=-3.7532 | p=0.0034 | 5% crit=-2.9064 -> STATIONARY (p < 0.05)


/tmp/ipykernel_37384/2135787266.py:21: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pval, _, _, crit, _ = adfuller(s)
/tmp/ipykernel_37384/2135787266.py:21: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pval, _, _, crit, _ = adfuller(s)
